# Time-domain first look — $\sigma^2(t)$ and $d(t)$ at the exit face (random slab 250×250×32 µm)
Narrow-band (FWHM 0.02 in $\nu$) **causal** reconstructions from the CW h5 (`scipy.fft.fft` over the ascending f-axis — Tidy3D stores $\hat E\propto\sum E\,e^{+2\pi i f t}$). Observation only: no fits, no $\xi$, no $D$. Conventions and verdicts: `README.md`, `bst_pipeline.py`.

In [ ]:
# --- constants & paths (experiment facts: README.md) --------------------------
import os, sys, gc
import numpy as np
import matplotlib.pyplot as plt
import tidy3d as td
from scipy.integrate import simpson
import scipy.fft
sys.path.append(os.path.abspath(r'../../../../../tidy3d'))
import AutomationModule as AM
plt.rc('font', family='Arial')

gap_data = AM.read_hdf5_as_dict(rf"../../20250630 MPB Bands analysis/Data/gap_data.hdf5")
transmission_data = AM.read_hdf5_as_dict("./data/slab_250x250x32/Transmission/LSU_20260804_slab_250x250x32_Transmission_background_n_1.00.h5")
a      = 2.562629142772549   # unit length a [um] (nu = a f/c)
L_slab = 32.0                # slab thickness [um] = 12.487 a
n_index      = "2.90"              # h5 group (n = 3.30)
GAP    = tuple(a/(14.3/(gap_data["Circular"]["0.22"]["gap_edges"][np.where(gap_data["Circular"]["0.22"]["n"]==float(n_index))[0]]).flatten()))      # MPB Data: gap edges for n=3.30, ff=0.22
print(f"n={n_index}, a={a:.6f} um, L_slab={L_slab:.3f} um, gap={GAP[0]:.4f}-{GAP[1]:.4f} (a/lambda)")
WORK   = "./random_slab_td"   # gitignored scratch (npz, pngs)
os.makedirs(WORK, exist_ok=True); os.makedirs("movies", exist_ok=True)

In [ ]:
# file_data = "./data/slab_250x250x32/LSU_20260804_slab_250x250x32_n_3p3__lsu_generated_healed_lambda_4_8.h5"
# file_data =  "./data/slab_250x250x32/LSU_20260804_slab_250x250x32_n_3p3_backdround_effective_n_1.34.h5"
# file_data =  "./data/slab_250x250x32/LSU_20260804_slab_250x250x32_n_3p3_backdround_effective_n_1.00.h5"
file_data = "./data/slab_250x250x32/LSU_20260804_slab_250x250x32_n_2p9_background_effective_n_1.00.h5"
# file_data = "./data/slab_250x250x32/LSU_20260804_slab_250x250x32_n_2p4_background_effective_n_1.00.h5"

In [ ]:
# --- load fields (~4.7 GB in RAM); axes; Simpson weights; sanity asserts -------
data = AM.read_hdf5_as_dict(file_data)
xg = np.asarray(data[n_index]["x"], float); yg = np.asarray(data[n_index]["y"], float)
f  = np.asarray(data[n_index]["f"], float)
nu = a*f/td.C_0                                    # ascending, 1500 bins
nx, ny, NFreqs = xg.size, yg.size, f.size
df    = float(np.mean(np.diff(f)))                 # 25.0 GHz
T_ps  = 1.0/df*1e12                                # FFT period = 40.0 ps (~ run_time!)
dt_ps = T_ps/NFreqs                                     # 26.7 fs
t  = np.arange(NFreqs)*dt_ps                            # causal time grid t_n = n dt
wxS = simpson(np.eye(nx), x=xg, axis=1)            # product-Simpson weights,
wyS = simpson(np.eye(ny), x=yg, axis=1)            # exact on the non-uniform grid
Xg, Yg = np.meshgrid(xg, yg, indexing="ij")
Rho2 = (Xg**2 + Yg**2).astype(np.float32)          # about the injection axis (0,0)
AW   = (wxS[:, None]*wyS[None, :]).astype(np.float32)
RIM  = np.sqrt(Xg**2 + Yg**2) > 45*a               # edge-loss artifact clock (rho > 45 a)
NRB  = 50                                          # 1a-wide annuli for encircled-power radii
rbin = np.minimum((np.sqrt(Rho2)/a).astype(int), NRB-1)
Mrad = np.zeros((NRB, nx*ny), np.float32)          # radial collector: row r = AW over annulus r
Mrad[rbin.ravel(), np.arange(nx*ny)] = AW.ravel()
A_rad  = Mrad.sum(1)                               # area per annulus
A2_rad = Mrad @ Rho2.ravel()                       # int rho^2 dA per annulus

def pr_diameter(P, I2):
    '''PR diameter d/a = 2 sqrt(PR/pi)/a, PR = P^2/I2 (background-robust).'''
    return 2*np.sqrt((P**2/I2)/np.pi)/a
for s in (5.0, 12.0, 25.0):                        # unit test: PR of a Gaussian = 4 sigma
    Ig = np.exp(-(Xg**2 + Yg**2)/(2*s**2))
    P_ = float((Ig*AW).sum()); I2_ = float((Ig**2*AW).sum())
    assert abs(pr_diameter(P_, I2_)*a/(4*s) - 1) < 1e-4
print(f"grid {nx}x{ny}; nu = [{nu[0]:.4f}, {nu[-1]:.4f}]; T = {T_ps:.3f} ps, dt = {dt_ps*1e3:.2f} fs; Gaussian PR test OK")

In [ ]:
# --- windows (FWHM 0.02) + one streaming pass: P, M2, I2, rim; Parseval; t-cutoffs ---
WINDOW = 0.005
SIG   = WINDOW/(2*np.sqrt(2*np.log(2)))              # Gaussian sigma_nu ~ 8.49e-3 (t_p ~ 0.16 ps)
DEEP  = {0.41, 0.43}                               # inside the PBG: measure the floor, not transport
FLANK = {GAP[0], GAP[1]}                       # blend across steep S(nu): qualitative only
# centres = sorted({0.33,0.35,GAP[0],GAP[1], (GAP[0]+GAP[1])/2, GAP[0]-WINDOW/2, GAP[0]-WINDOW, GAP[1]+WINDOW/2, GAP[1]+WINDOW,0.45,0.48})
centres = sorted({GAP[0]-0.02, GAP[0]-WINDOW/2-0.001, GAP[1]+WINDOW/2+0.001,GAP[1]+0.02})
def classify(c, gap=GAP, sig=SIG, k=1.0):
    lo, hi = gap
    if lo + k*sig <= c <= hi - k*sig:
        return 'deep'                      # window fully inside the gap
    if (lo - k*sig) <= c <= (hi + k*sig):
        return 'flank'                     # window straddles an edge
    return 'ok'
kind = [classify(c) for c in centres]
W  = [np.exp(-0.5*((nu-c)/SIG)**2).astype(np.float32) for c in centres]
nW = len(centres)
# time-smoothing kernel (needed by the streaming pass for the smooth-then-square I2sm):
# each window = probe pulse with Gaussian *intensity* envelope sigma_tI = 1/(2 pi sigma_f sqrt(2))
from scipy.ndimage import gaussian_filter1d
SMOOTH = 1                                         # kernel width in pulse-envelope units
sig_tI = 1e12/(2*np.pi*(SIG*td.C_0/a)*np.sqrt(2))    # intensity-envelope sigma [ps]
n_smp  = SMOOTH*sig_tI/dt_ps                         # kernel sigma in samples
print(kind,centres)
print(f"smoothing kernel: sigma_t(I) = {SMOOTH*sig_tI*1e3:.0f} fs "
      f"(FWHM {SMOOTH*sig_tI*2.355:.2f} ps, {n_smp:.1f} samples)")

In [ ]:
CLIP = 1e-12                                        # strip I < CLIP * frame max (per window, per time frame)
P  = np.zeros((nW, NFreqs)); #total power (zeroth moment, the normalizer)
M2 = np.zeros((nW, NFreqs)); #second spatial moment of the intensity
I2 = np.zeros((nW, NFreqs)); #fourth spatial moment of the intensity
I2sm = np.zeros((nW, NFreqs)); #int (time-smoothed I)^2 dA -> unbiased smoothed PR
P_unc = np.zeros((nW, NFreqs)); #unclipped power (Parseval check only)
#σ²(t) = M2/P
Prim = np.zeros((nW, NFreqs)); spec = np.zeros(nW) #(rim power) is a diagnostic 
#accumulator: the summed intensity in the outermost ring of pixels of the transverse window — e.g. the last few rows/columns at the edges of the monitor.
Rhist = np.zeros((nW, NRB, NFreqs), np.float32)    # radial power histogram (1a annuli) per frame
Ex_, Ey_, Ez_ = data[n_index]["Ex"], data[n_index]["Ey"], data[n_index]["Ez"]
s16 = np.float32(1e16)                             # rescale at read: I ~ O(1), I^2 cannot underflow f32
BL = 33 #Block length to save memory nx = ny = 363 = 11×33
R2AW = (Rho2*AW).astype(np.float32)
IwF = np.empty((nx, ny, NFreqs), np.float32)       # full-plane movie for ONE window (~0.8 GB):
                                                   # the per-frame max needs the whole plane before clipping,
                                                   # so windows are now the OUTER loop, blocks inner
for k in range(nW):
    for i0 in range(0, nx, BL):                    # fill the movie block by block (same FFTs as before)
        i1 = min(i0+BL, nx)
        bx = np.asarray(Ex_[i0:i1, :, 0, :])*s16   # (nb, ny, N) complex64
        by = np.asarray(Ey_[i0:i1, :, 0, :])*s16
        bz = np.asarray(Ez_[i0:i1, :, 0, :])*s16
        IwF[i0:i1]  = np.abs(scipy.fft.fft(bx*W[k], axis=-1, workers=-1))**2
        IwF[i0:i1] += np.abs(scipy.fft.fft(by*W[k], axis=-1, workers=-1))**2
        IwF[i0:i1] += np.abs(scipy.fft.fft(bz*W[k], axis=-1, workers=-1))**2   # causal I_W(x,y,t)
        Ispec = np.abs(bx)**2 + np.abs(by)**2 + np.abs(bz)**2
        spec[k] += float((np.tensordot(Ispec, (W[k]**2).astype(np.float64), axes=([2], [0]))*AW[i0:i1]).sum())
        del bx, by, bz, Ispec
    P_unc[k] = np.einsum('xyn,xy->n', IwF, AW, dtype=np.float64)     # pre-clip (Parseval)
    thr = (CLIP*IwF.max(axis=(0, 1))).astype(np.float32)             # per-frame threshold
    IwF[IwF < thr[None, None, :]] = 0.0                              # strip background/noise floor
    P[k]   = np.einsum('xyn,xy->n', IwF, AW, dtype=np.float64)
    M2[k]  = np.einsum('xyn,xy->n', IwF, R2AW, dtype=np.float64)
    I2[k]  = np.einsum('xyn,xy->n', IwF**2, AW, dtype=np.float64)
    Iws = gaussian_filter1d(IwF, n_smp, axis=-1, mode='wrap')        # smooth I, THEN square
    I2sm[k] = np.einsum('xyn,xy->n', Iws**2, AW, dtype=np.float64)
    del Iws
    Prim[k] = np.einsum('pn,p->n', IwF[RIM], AW[RIM], dtype=np.float64)
    Rhist[k] = Mrad @ IwF.reshape(nx*ny, NFreqs)                     # radial distribution of the (clipped) movie
    gc.collect()
    print(f"window {k+1}/{nW}", end="\r")
del IwF; gc.collect()
print()
parseval = P_unc.sum(1)/(NFreqs*spec)                   # DFT Parseval on the UNCLIPPED power
assert np.abs(parseval - 1).max() < 1e-2, parseval
clip_loss = 1 - P.sum(1)/P_unc.sum(1)                   # energy removed by the CLIP floor
sig2 = (M2/np.maximum(P, 1e-300))/a**2             # sigma^2(t) [a^2] about (0,0)
d_t  = pr_diameter(P, np.maximum(I2, 1e-300))      # d(t) [a]
rim_frac = Prim/np.maximum(P, 1e-300)

# wrap-around / floor diagnostic (run_time ~ T: undecayed tails fold to t ~ 0+) + validity cutoff
i_pk  = P.argmax(1); P_pk = P.max(1)
floor = np.median(P[:, t > T_ps-2.0], axis=1)      # late-time floor level
end_flat = np.median(P[:, t > T_ps-1.0], 1)/np.maximum(np.median(P[:, (t > T_ps-6.0) & (t < T_ps-5.0)], 1), 1e-300)
early_db = 10*np.log10(np.maximum(P[:, t < 0.10].max(1), 1e-300)/P_pk)
t_floor = np.full(nW, T_ps); t_rim = np.full(nW, T_ps)
for k in range(nW):
    j = np.nonzero(P[k, i_pk[k]:] < 3*floor[k])[0]         # P sinks into the floor
    if j.size: t_floor[k] = t[i_pk[k]+j[0]]
    j = np.nonzero((rim_frac[k] > 0.02) & (P[k] > 1e-3*P_pk[k]))[0]   # rim clock: halo at absorbers (first crossing;
    # conservative for 0.389/0.39/0.43: a 0.3-1 ps rim transient precedes an otherwise rim-clean quasi-mode segment)
    if j.size: t_rim[k] = t[j[0]]
t_valid = np.minimum(t_floor, t_rim)
t_arr = np.zeros(nW)                                       # arrival gate: first P > 1e-3 P_pk
for k in range(nW):
    j = np.nonzero(P[k] > 1e-3*P_pk[k])[0]
    if j.size: t_arr[k] = t[j[0]]
floor_db = 10*np.log10(floor/P_pk)
print(" nu_0   kind    t_arr  t_pk[ps]  floor[dB]  end/mid  early[dB]  t_floor  t_rim  t_valid[ps]")
for k in range(nW):
    print(f" {centres[k]:.3f}  {kind[k]:5s}  {t_arr[k]:5.2f}  {t[i_pk[k]]:8.2f}  {floor_db[k]:8.1f}  "
          f"{end_flat[k]:7.2f}  {early_db[k]:8.1f}  {t_floor[k]:7.2f}  {t_rim[k]:6.2f}  {t_valid[k]:6.2f}")
ok = np.array([kd == 'ok' for kd in kind])
bnd = np.maximum(floor_db, early_db)                       # signal level at the t~0/T boundary, dB re window peak
bad = [f"{centres[k]:.3f}" for k in range(nW) if bnd[k] > -40]
print(f"CUTOFF: trust t_arr < t < t_valid = min(P-into-floor, rim>2%); ok-window median t_valid = {np.median(t_valid[ok]):.1f} ps.")
print(f"WRAP/FLOOR: run_time ~ T, so undecayed tails fold to t~0+. Boundary (t~0/T) level is within -40 dB of the window "
      f"peak for [{', '.join(bad)}]: spectrum-edge windows (clipped Gaussian -> aliased end sidelobe), in-gap windows "
      f"(floor/quasi-modes), and slow-decaying gap-adjacent windows whose end tail still decays (tau ~ 6-7 ps -> folded "
      f"copy <= 0.3% of P on trusted segments). Elsewhere the boundary is <= -40 dB: fold-back negligible below t_valid.")
print(f"Parseval max |ratio-1| = {np.abs(parseval-1).max():.2e}; "
      f"CLIP={CLIP:.0e} strips at most {100*clip_loss.max():.2f}% of a window's total energy")
# np.savez(os.path.join(WORK, "td_curves.npz"), t=t, centres=np.array(centres), kind=np.array(kind),
#          P=P, M2=M2, I2=I2, Prim=Prim, sig2=sig2, d_t=d_t, rim_frac=rim_frac, parseval=parseval,
#          t_valid=t_valid, t_floor=t_floor, t_rim=t_rim, t_arr=t_arr, floor=floor)

In [ ]:
# --- smooth over the analysis-pulse duration (temporal resolution of the window) ---
# Kernel (SMOOTH, n_smp) is defined with the windows and already used by the
# streaming pass for I2sm = int (time-smoothed I)^2 dA. Here: convolve the
# *linear* accumulators P, M2 with the same envelope (mode='wrap': the DFT
# reconstruction is T-periodic) and re-form the ratios -> exactly the moments
# of the time-smoothed intensity movie, power-weighted so sigma^2 stays tame
# where P is small. d uses I2sm (smooth-THEN-square), i.e. the PR *of* the
# smoothed movie -- no low bias from within-kernel beating variance.
Ps, M2s = (gaussian_filter1d(A, n_smp, axis=1, mode='wrap') for A in (P, M2))
sig2_s = (M2s/np.maximum(Ps, 1e-300))/a**2          # smoothed sigma^2(t) [a^2]
d_t_s  = pr_diameter(Ps, np.maximum(I2sm, 1e-300))  # smoothed d(t) [a], unbiased
print(f"smoothing kernel: sigma_t(I) = {SMOOTH*sig_tI*1e3:.0f} fs "
      f"(FWHM {SMOOTH*sig_tI*2.355:.2f} ps, {n_smp:.1f} samples)")

In [ ]:
# --- core-faithful widths: encircled-power radii + pedestal-subtracted sigma^2 ---
# sigma^2 = M2/P weights every photon by rho^2, so a dim slowly-decaying pedestal
# (wrap-around fold, rim leakage, noise floor) dominates once P(t) has decayed a
# few decades -> sigma^2 creeps upward even when the bright core (what the
# log-scale movies show) is frozen. Conversely a quasi-static wide pedestal fakes
# a plateau (the effective-cladding run, README section 10). Both estimators below
# are calibrated to EQUAL the Plot-1 sigma^2 = <rho^2> (= 2 sigma_axis^2) for a
# Gaussian profile, but read only the core:
#   rho_q   -> radius enclosing fraction q of the pedestal-subtracted power
#   sig2_bs -> exact second moment after removing a flat pedestal fitted on the
#              45-48a annuli (inside the inscribed circle). Past t_rim those annuli
#              carry real signal and sig2_bs biases LOW -> trust only t < t_valid.
Rs  = gaussian_filter1d(Rhist, n_smp, axis=-1, mode='wrap')     # smoothed radial power
bkg = Rs[:, 45:48, :].sum(1)/A_rad[45:48].sum()                 # flat-pedestal density per frame
Rs_bs = np.maximum(Rs - bkg[:, None, :]*A_rad[None, :, None], 0.0)
cum = np.cumsum(Rs_bs, axis=1)
def rho_q(c, q):
    '''radius enclosing fraction q of the power (linear interp inside 1a annuli) [um]'''
    tgt = q*c[:, -1, :]
    idx = (c >= tgt[:, None, :]).argmax(1)
    c_hi = np.take_along_axis(c, idx[:, None, :], 1)[:, 0]
    c_lo = np.where(idx > 0, np.take_along_axis(c, np.clip(idx-1, 0, NRB-1)[:, None, :], 1)[:, 0], 0.0)
    return (idx + np.clip((tgt - c_lo)/np.maximum(c_hi - c_lo, 1e-300), 0, 1))*a
sig2_50 = (rho_q(cum, 0.50)/a)**2/np.log(2)        # == <rho^2> [a^2] for a Gaussian profile
sig2_90 = (rho_q(cum, 0.90)/a)**2/np.log(10)       # == <rho^2> [a^2] for a Gaussian profile
Pbs, M2bs = Ps - bkg*A_rad.sum(), M2s - bkg*A2_rad.sum()
sig2_bs  = np.where(Pbs > 0.2*Ps, M2bs/np.maximum(Pbs, 1e-300), np.nan)/a**2
ped_frac = 1 - Pbs/np.maximum(Ps, 1e-300)          # pedestal power fraction (diagnostic)
# unit test: Gaussian sigma_axis = 12 um -> <rho^2> = 288 um^2 through the same machinery
_cg = np.cumsum(Mrad @ np.exp(-Rho2/(2*12.0**2)).ravel(), 0)[None, :, None]
for _s2 in ((rho_q(_cg, .5)/a)**2/np.log(2), (rho_q(_cg, .9)/a)**2/np.log(10)):
    assert abs(_s2.item()*a**2/(2*12.0**2) - 1) < 0.05, _s2
print(" nu_0: pedestal power fraction at t_valid (should be small where curves are trusted)")
print(" " + "  ".join(f"{centres[k]:.3f}:{np.interp(t_valid[k], t, ped_frac[k]):.2f}" for k in range(nW)))

In [ ]:
# --- S(nu) transmission proxy (same object as 20260806_Beam_Diameter_d_nu) -----
# S1 = int |E|^2 dA per bin, divided by a smooth envelope fit through the bins
# OUTSIDE the dip (raw DFT carries the source envelope -> only RELATIVE features
# are physical, not a calibrated T). Used as a side panel next to the t-plots.

try:
    trans = transmission_data[n_index]["Transmission_Exit"]
    nu_trans = a*transmission_data[n_index]["f"]/td.C_0
except NameError:
    S1 = np.zeros(NFreqs)
    for i0 in range(0, nx, BL):
        i1 = min(i0+BL, nx)
        Ib = (np.abs(Ex_[i0:i1, :, 0, :])**2 + np.abs(Ey_[i0:i1, :, 0, :])**2
              + np.abs(Ez_[i0:i1, :, 0, :])**2).astype(np.float64)
        S1 += np.einsum("xyn,xy->n", Ib, AW[i0:i1].astype(np.float64))
        del Ib
    gc.collect()
    mask_out = (nu < 0.36) | (nu > 0.47)                 # envelope ref excludes the dip
    S_ref = 10**np.polyval(np.polyfit(nu[mask_out], np.log10(S1[mask_out]), 7), nu)
    trans = S1/S_ref      
    nu_trans = nu                               # relative transmission proxy

## $\sigma^2(t)$ and $d(t)$ — solid = trusted ($t_{arr}<t<t_{valid}$); faded = pre-arrival/beyond cutoff; flank dashed (qualitative); deep gap grey (not beam transport)

In [ ]:
# --- Plot 1: sigma^2(t) [a^2] vs t [ps], with S(nu) side panel -----------------
from matplotlib import colormaps, cm, colors
from matplotlib.lines import Line2D
norm = colors.Normalize(vmin=min(centres), vmax=max(centres))
cmap = colormaps['copper']

def plot_curves(Y, ylabel, fname, Yraw=None):
    fig, (ax, axs) = plt.subplots(1, 2, figsize=(11.8, 5.2), dpi=110,
                                  gridspec_kw={'width_ratios': [3.4, 1], 'wspace': 0.06})
    for k in range(nW):
        if kind[k] == 'deep':
            ax.plot(t, Y[k], color='0.55', ls=':', lw=1.0, alpha=0.6, zorder=1,label=rf"$\nu_0=${centres[k]:.4f}")
            continue
        col = cmap(norm(centres[k])); ls = '--' if kind[k] == 'flank' else '-'
        v = (t >= t_arr[k]) #& (t <= t_valid[k])             # solid: post-arrival, pre-cutoff
        if Yraw is not None:                                 # raw curve, faint, behind the smoothed one
            ax.plot(t[v], Yraw[k][v], color=col, ls='-', lw=0.6, alpha=0.2, zorder=1)
        ax.plot(t[v], Y[k][v], color=col, ls=ls, lw=1.6, zorder=2, label=rf"$\nu_0=${centres[k]:.4f}")
        # ax.plot(t[~v], Y[k][~v], color=col, ls=ls, lw=0.7, alpha=0.15)
    vtr = np.concatenate([Y[k][(t >= t_arr[k]) & (t <= t_valid[k])] for k in range(nW) if kind[k] != 'deep'])
    ymax = 1.3*np.nanmax(vtr) if np.isfinite(vtr).any() else 1.0   # NaN-tolerant (sig2_bs masks frames)
    xmax = min(T_ps, 65
               #1.10*max(t_valid[k] for k in range(nW) if kind[k] != 'deep')
               )
    ax.set_xlim(0, xmax); ax.set_ylim(0, ymax)
    ax.set_xlabel('t [ps]'); ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25, lw=0.5)
    # --- side panel: transmission(nu) (20260806 convention), nu vertical, windows as bands ---
    # each window = a shaded horizontal band nu_0 +/- FWHM/2 in the curve's colour
    axs.plot(trans, nu_trans, lw=0.8, color='m', zorder=3)
    axs.set_xscale('log'); axs.set_xlim(1, np.min(trans)*1e-2); axs.set_ylim(np.min(nu_trans), np.max(nu_trans))
    axs.axhspan(GAP[0], GAP[1], color='gray', alpha=0.9, zorder=0)
    for c, kd in zip(centres, kind):
        colc = '0.45' if kd == 'deep' else cmap(norm(c))
        axs.axhspan(c - WINDOW/2, c + WINDOW/2, color=colc, alpha=0.4, lw=0, zorder=1)
    axs.yaxis.tick_right(); axs.yaxis.set_label_position('right')
    axs.set_xlabel(r'Transmission'); axs.set_ylabel(r'$\nu = a/\lambda$')
    axs.grid(alpha=0.25, lw=0.5, which='both')
    # ax.legend(handles=[Line2D([], [], color='k', ls='-', label=r'valid ($t_{arr}<t<t_{valid}$)'),
    #                    Line2D([], [], color='k', ls='-', lw=0.7, alpha=0.25, label='pre-arrival / beyond cutoff'),
    #                    Line2D([], [], color='k', ls='--', label='flank window (qualitative)'),
    #                    Line2D([], [], color='0.55', ls=':', label='deep gap (not beam transport)')],
    #           loc='upper right', fontsize=9, framealpha=0.9)
    # 
    ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
    fig.tight_layout(); fig.savefig(os.path.join(WORK, fname), dpi=140); plt.show()
    return ax

plot_curves(sig2_s, r'$\sigma^2(t) = \int\rho^2 I\,dA\,/\int I\,dA\;\;[a^2]$', 'sigma2_t.png');

In [ ]:
# --- Plot 2: PR diameter d(t) [a] vs t [ps] ------------------------------------
plot_curves(d_t_s, r'$d(t) = 2\sqrt{A_{eff}/\pi}\;\;[a]$', 'd_t.png');

## Exit-face movies $\log_{10}(I/I_{max})$, fixed normalization, every 5th frame → `movies/`

In [ ]:
# # --- movies: 4 selected windows, decimated x5 (~300 frames each) ---------------

# from PIL import Image
# MOVIE_NUS = centres
# # LABEL = {0.35: 'below gap', 0.389: 'lower flank dip', 0.437: 'upper flank dip', 0.55: 'mid-band'}
# DEC = 5
# t_dec = t[::DEC]; Nt = t_dec.size
# carr = np.array(centres)
# midx = [int(np.argmin(np.abs(carr - c))) for c in MOVIE_NUS]
# movs = np.zeros((len(MOVIE_NUS), nx, ny, Nt), np.float32)          # ~160 MB per window
# for i0 in range(0, nx, BL):
#     i1 = min(i0+BL, nx)
#     bx = np.asarray(Ex_[i0:i1, :, 0, :])*s16
#     by = np.asarray(Ey_[i0:i1, :, 0, :])*s16
#     bz = np.asarray(Ez_[i0:i1, :, 0, :])*s16
#     for m, k in enumerate(midx):
#         Iw  = np.abs(scipy.fft.fft(bx*W[k], axis=-1, workers=-1))**2
#         Iw += np.abs(scipy.fft.fft(by*W[k], axis=-1, workers=-1))**2
#         Iw += np.abs(scipy.fft.fft(bz*W[k], axis=-1, workers=-1))**2
#         movs[m, i0:i1] = Iw[:, :, ::DEC]
#     del bx, by, bz, Iw; gc.collect()
#     print(f"rows {i1}/{nx}", end="\r")
# print()


In [ ]:
# movs.shape

In [ ]:
# from matplotlib.colors import LogNorm
# for m, (c, k) in enumerate(zip(MOVIE_NUS, midx)):
#     I = (np.clip(movs[m]/movs[m].max(axis=(0,1)), 1e-2, None))    # fixed global norm, 1e-2 floor
#     fig, ax = plt.subplots(figsize=(5.4, 4.6), dpi=88)
#     qm = ax.pcolormesh(xg, yg, I[:, :, 0].T, shading='nearest', cmap='inferno',norm=LogNorm(vmin=1e-2, vmax=1))
#     ax.set_aspect('equal'); ax.set_xlabel('x [um]'); ax.set_ylabel('y [um]')
#     fig.colorbar(qm, ax=ax, label=r'$\log_{10}(I/I_{max})$')
#     fig.tight_layout()
#     frames = []
#     for j in range(Nt):
#         qm.set_array(I[:, :, j].T)
#         ax.set_title(f"$\\nu_0$ = {carr[k]:.3f}   t = {t_dec[j]:5.2f} ps")
#         fig.canvas.draw()
#         frames.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[:, :, :3].copy()))
#     plt.close(fig)
#     name = "movies/beam_spreading_nu" + f"{carr[k]:.3f}".replace('.', 'p') + ".gif"
#     frames[0].save(name, save_all=True, append_images=frames[1:], duration=50, loop=0)
#     print(f"{name}  ({os.path.getsize(name)/1e6:.1f} MB, {Nt} frames)")
# del movs, frames; gc.collect()

## Exit-face speckle point cloud in $(x, y, t)$ — Yamilov et al., *Nat. Phys.* **19** (2023), Fig. 4c style
At each sampled time step inside $[t_0,t_1]$, the speckle grains with $I > \mathrm{THRESH}\cdot I_{max}(t)$ are scattered as points at $(t_s, x, y)$, colored by $\log_{10}(I/I_{max}(t))$ — the cloud envelope (expanding cone vs frozen cylinder) is the observable. Per-frame normalization because the total power decays over decades across the window. One cloud per window $\nu_c$ (dropdown). Mind $t_{arr}$/$t_{valid}$ per window (printed below).

In [ ]:
# --- Fig 4c-style: speckle point cloud in (x, y, t), per window -----------------
import plotly.graph_objects as go

T_GATE   = (0.0, 50.0) # time window [t0, t1] in ps -- edit and re-run
N_SLICES = 150         # time steps sampled inside the window (>= frame count -> every frame)
# THRESH is relative to the FRAME MAX. THRESH = LOG_FLOOR drops exactly the
# points that would render at the colormap floor (black/grey shell) and nothing
# the color scale can express; THRESH = 0 keeps every pixel (envelope = whole
# monitor, structure in color only); larger values isolate the bright core.
LOG_FLOOR = 1e-3       # color floor for I/I_max(t)
THRESH    = LOG_FLOOR  # keep grid points with I > THRESH * I_max(t) (per frame)
MAX_PTS   = 1000       # random cap per frame (unbiased subsample); None = no cap
save_fig=False

j0, j1 = np.searchsorted(t, T_GATE[0]), np.searchsorted(t, T_GATE[1])
assert j1 > j0, "empty time window"
js = np.unique(np.linspace(j0, j1-1, N_SLICES).round().astype(int))
nS = js.size
print(f"{nS} slices, t = {t[js[0]]:.2f}..{t[js[-1]]:.2f} ps; per-window trusted range:")
for k in range(nW):
    tag = " <-- window outside trusted range" if (T_GATE[1] < t_arr[k] or T_GATE[0] > t_valid[k]) else ""
    print(f"  nu={centres[k]:.3f} ({kind[k]:5s}): t_arr={t_arr[k]:.2f}, t_valid={t_valid[k]:.2f} ps{tag}")

Islc = np.zeros((nW, nS, nx, ny), np.float32)      # sampled speckle frames (~0.3 GB at 150 slices)
for i0 in range(0, nx, BL):
    i1 = min(i0+BL, nx)
    bx = np.asarray(Ex_[i0:i1, :, 0, :])*s16
    by = np.asarray(Ey_[i0:i1, :, 0, :])*s16
    bz = np.asarray(Ez_[i0:i1, :, 0, :])*s16
    for k in range(nW):
        Iw  = np.abs(scipy.fft.fft(bx*W[k], axis=-1, workers=-1)[..., js])**2
        Iw += np.abs(scipy.fft.fft(by*W[k], axis=-1, workers=-1)[..., js])**2
        Iw += np.abs(scipy.fft.fft(bz*W[k], axis=-1, workers=-1)[..., js])**2
        Islc[k, :, i0:i1] = np.moveaxis(Iw, -1, 0)
    del bx, by, bz, Iw; gc.collect()
    print(f"rows {i1}/{nx}", end="\r")
print()

# point cloud: for each frame keep grains above THRESH of the frame max,
# scatter them at (t_s, x, y), colored by I/I_max(t) on a log scale, dim first.
# Scatter3d has no native LogNorm: color internally by log10(I) and relabel the
# colorbar ticks as powers of ten -> reads as a lognorm intensity bar.
rng = np.random.default_rng(0)
def cloud(k):
    tp, xp, yp, cp = [], [], [], []
    for s in range(nS):
        In = Islc[k, s]/Islc[k, s].max()
        ii, jj = np.nonzero(In > THRESH)
        if MAX_PTS is not None and ii.size > MAX_PTS:
            sel = rng.choice(ii.size, MAX_PTS, replace=False)
            ii, jj = ii[sel], jj[sel]
        tp.append(np.full(ii.size, t[js[s]])); xp.append(xg[ii]/a); yp.append(yg[jj]/a)
        cp.append(np.log10(np.clip(In[ii, jj], LOG_FLOOR, None)))
    tp, xp, yp, cp = (np.concatenate(v) for v in (tp, xp, yp, cp))
    o = np.argsort(cp)
    return tp[o], xp[o], yp[o], cp[o]

dec0 = int(np.floor(np.log10(LOG_FLOOR)))
tickv = list(range(dec0, 1))                       # e.g. -3, -2, -1, 0
tickt = [f"10<sup>{v}</sup>" if v else "1" for v in tickv]
fig3d = go.Figure()
for k in range(nW):
    tp, xp, yp, cp = cloud(k)
    fig3d.add_scatter3d(x=tp, y=xp, z=yp, mode='markers', visible=(k == 0),
                        marker=dict(size=1.6, opacity=0.5, color=cp, colorscale='Inferno',
                                    cmin=dec0, cmax=0,
                                    colorbar=dict(title='Intensity(t)', len=0.6,
                                                  tickvals=tickv, ticktext=tickt)),
                        customdata=10.0**cp[:, None],
                        hovertemplate=('t = %{x:.2f} ps<br>x = %{y:.1f} a<br>y = %{z:.1f} a'
                                       '<br>Intensity(t) = %{customdata[0]:.2e}<extra></extra>'))
def _title(k):
    return (f"nu_0 = {centres[k]:.3f}"
            f"Time Window: {T_GATE[0]:.1f}-{T_GATE[1]:.1f} ps")
fig3d.update_layout(
    title=_title(0), width=900, height=620, margin=dict(l=0, r=0, t=60, b=0),
    scene=dict(xaxis_title='t [ps]', yaxis_title='x [a]', zaxis_title='y [a]',
               yaxis=dict(range=[xg[0]/a, xg[-1]/a]), zaxis=dict(range=[yg[0]/a, yg[-1]/a]),
               aspectratio=dict(x=2.2, y=1, z=1),
               camera=dict(eye=dict(x=2.1, y=1.5, z=0.9))),
    updatemenus=[dict(buttons=[dict(label=f"nu = {centres[k]:.3f} ({kind[k]})", method='update',
                                    args=[{'visible': [i == k for i in range(nW)]},
                                          {'title': _title(k)}])
                               for k in range(nW)],
                      direction='down', x=1.0, xanchor='right', y=1.08, yanchor='top')])
# camera button in the modebar: downloads the CURRENT view (camera, dropdown
# selection) as a 3x-resolution PNG -> browser Downloads folder. plotly.js
# cannot export PDF client-side; png_to_pdf() below wraps the download into one.
CFG = {'toImageButtonOptions': {'format': 'png', 'scale': 3,
                                'filename': f"exit_face_cloud_{T_GATE[0]:.0f}-{T_GATE[1]:.0f}ps"},
       'displaylogo': False}
fig3d.show(config=CFG)

def png_to_pdf(png_path, pdf_path=None):
    '''wrap a downloaded PNG (unchanged, full resolution) into a single-page PDF'''
    from PIL import Image
    pdf_path = pdf_path or os.path.splitext(png_path)[0] + ".pdf"
    Image.open(png_path).convert("RGB").save(pdf_path, resolution=300)
    return pdf_path
# e.g.: png_to_pdf(os.path.expanduser(r"~\Downloads\exit_face_cloud_3-40ps.png"))

# --- persist: standalone interactive HTML + slice data (npz) --------------------
if save_fig:
    gtag = f"{n_index}_{T_GATE[0]:.1f}-{T_GATE[1]:.1f}ps".replace('.', 'p')
    fig3d.write_html(os.path.join(WORK, f"exit_face_cloud_{gtag}.html"), include_plotlyjs=True, config=CFG)
    np.savez(os.path.join(WORK, f"exit_face_slices_{gtag}.npz"), Islc=Islc, t_slices=t[js],
             xg=xg, yg=yg, a=a, centres=np.array(centres), kind=np.array(kind), T_GATE=np.array(T_GATE))
    print(f"saved exit_face_cloud_{gtag}.html + exit_face_slices_{gtag}.npz in {WORK}")